# ManualMate — answer eval on Kaggle (week 3)

Same code as Colab section 7d. Use this when Colab has no free GPU.

Before you run (right panel → **Settings**, or **Add-ons** at the top):
1. **Accelerator → GPU T4 x2**.
2. **Internet → On** (needs a verified phone number).
3. **Add-ons → Secrets**: add `DATABASE_URL` and `GEMINI_API_KEY` (value only: no quotes, no `NAME=`, no extra line). Tick **Attach** for both.

Run the cells one by one, top to bottom.

## 1. Check the GPU
Expected: lines with `Tesla T4`.

In [ ]:
!nvidia-smi -L

## 2. Get the latest code from GitHub
Expected: the last commit, for example `91a6ad8 Add answer eval ...` or newer.

In [ ]:
import os
if not os.path.exists("/kaggle/working/manualmate"):
    !git clone https://github.com/furkanhf/manualmate.git /kaggle/working/manualmate
%cd /kaggle/working/manualmate
!git pull -q
!git log --oneline -1

## 3. Install the libraries
Kaggle already has PyTorch with GPU support, so we do not install it. Pip may print dependency warnings: they are harmless.

In [ ]:
!pip install -q sentence-transformers==6.2.0 rank-bm25==0.2.2 "psycopg[binary]" pgvector python-dotenv google-genai
!pip install -q -e . --no-deps

## 4. Load the secrets
Expected: `DATABASE_URL loaded`, `GEMINI_API_KEY loaded`. The values are never printed.

In [ ]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for name in ["DATABASE_URL", "GEMINI_API_KEY"]:
    os.environ[name] = secrets.get_secret(name).strip()
    print(name, "loaded")

## 5. Quick test (3 questions per set, ~2 min with the model downloads)
Expected: lines like `q001  answer  ok  score=0.9...`, then `===== SUMMARY`.

In [ ]:
!python scripts/run_answer_eval.py --limit 3

## 6. Full run (50 + 30 + 10 questions, ~80 Gemini calls, about 5–8 minutes)

In [ ]:
!python scripts/run_answer_eval.py

## 7. Download the new results
Click each link to download. Save the files in `D:\\Projects\\manualmate\\eval\\results`.

In [ ]:
import subprocess
from IPython.display import FileLink, display
new = subprocess.run(["git", "ls-files", "--others", "--exclude-standard", "eval/results"],
                     capture_output=True, text=True).stdout.split()
print(len(new), "new files")
for path in new:
    display(FileLink(path))